<a href="https://colab.research.google.com/github/03sarath/Agent2Agent/blob/main/competitive-intel/notebooks/competitive_intelligence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Psitron: Enterprise Competitive Intelligence Platform
### Multi-Agent System using Google ADK + Vertex AI

This notebook builds a **4-agent competitive intelligence pipeline** step by step.

**What you will build:**

| Step | Agent | What it does |
|------|-------|-------------|
| 1 | Market Scanner | Finds recent news, moves, announcements |
| 2 | Sentiment Analyzer | Scores brand perception from the web |
| 3 | Pricing Intelligence | Extracts competitor pricing data |
| 4 | Report Generator | Synthesizes an executive brief |

**Architecture:**
```
User Query
    │
    ▼
SequentialAgent (host)
    ├── market_scanner      → google_search
    ├── sentiment_analyzer  → google_search
    ├── pricing_intelligence → google_search
    └── report_generator    (no tools — synthesizes from context)
```
All agents are simple `Agent()` objects — no custom classes needed.

## Section 1 — Install Dependencies

In [ ]:
%pip install --upgrade -q google-adk==1.9.0 google-genai google-cloud-aiplatform[agent_engines,adk] python-dotenv nest_asyncio

## Section 2 — Configure Google Cloud

In [ ]:
import os
import sys

# ── Set your GCP project details here ────────────────────────────────────────
GOOGLE_CLOUD_PROJECT  = "your-gcp-project-id"   # <-- change this
GOOGLE_CLOUD_LOCATION = "us-central1"

# Tell ADK to use Vertex AI instead of Google AI Studio
os.environ["GOOGLE_GENAI_USE_VERTEXAI"]  = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"]       = GOOGLE_CLOUD_PROJECT
os.environ["GOOGLE_CLOUD_LOCATION"]      = GOOGLE_CLOUD_LOCATION

# ── Authenticate (Colab only) ─────────────────────────────────────────────────
if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user(project_id=GOOGLE_CLOUD_PROJECT)

print("Configuration:")
print(f"  Project  : {GOOGLE_CLOUD_PROJECT}")
print(f"  Location : {GOOGLE_CLOUD_LOCATION}")
print(f"  Backend  : Vertex AI")

## Section 3 — Imports

In [ ]:
import asyncio
import nest_asyncio

from google.adk.agents import Agent, SequentialAgent
from google.adk.artifacts import InMemoryArtifactService
from google.adk.memory.in_memory_memory_service import InMemoryMemoryService
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search
from google.genai import types

# Allow async code to run inside Jupyter / Colab
nest_asyncio.apply()

print("All imports successful!")

## Section 4 — Define the 4 Specialist Agents

Each agent is a plain `Agent()` call — no custom classes.  
The `instruction` tells the agent exactly what to do and what format to return.

In [ ]:
MODEL = "gemini-2.5-pro"

# ── Agent 1: Market Scanner ───────────────────────────────────────────────────
market_scanner = Agent(
    model=MODEL,
    name="market_scanner",
    instruction="""
You are a market intelligence specialist. Scan the web for recent developments
about the competitor mentioned in the conversation.

Steps:
1. Read the conversation to identify the competitor name.
2. Search for the competitor name + "news 2025", then + "product launch", then + "announcement".
3. Focus on the last 30-60 days only.
4. Extract: new products, partnerships, executive changes, funding rounds,
   geographic expansion, and strategic shifts.

Return a structured section titled **MARKET SCAN** with:
- Recent News (bullet list, include dates where available)
- Key Strategic Moves
- Notable Announcements
""",
    tools=[google_search],
)

print("✅ Agent 1 created: market_scanner")

In [ ]:
# ── Agent 2: Sentiment Analyzer ──────────────────────────────────────────────
sentiment_analyzer = Agent(
    model=MODEL,
    name="sentiment_analyzer",
    instruction="""
You are a brand sentiment analyst. Analyze public perception of the competitor
that was identified earlier in this conversation.

Steps:
1. Read the conversation to identify the competitor name.
2. Search for the competitor name + "reviews 2025", then + "complaints", then + "customer feedback", then + "analyst opinion".
3. Look for patterns across customers, analysts, press, and social media.

Return a structured section titled **SENTIMENT ANALYSIS** with:
- Overall Sentiment: Positive / Neutral / Negative  (score 1-10)
- Top Positive Themes (what people praise)
- Top Negative Themes (complaints, concerns)
- Analyst & Media Perception
""",
    tools=[google_search],
)

print("✅ Agent 2 created: sentiment_analyzer")

In [ ]:
# ── Agent 3: Pricing Intelligence ────────────────────────────────────────────
pricing_intelligence = Agent(
    model=MODEL,
    name="pricing_intelligence",
    instruction="""
You are a pricing intelligence analyst. Find and analyze the competitor's
pricing strategy based on what was discussed earlier in this conversation.

Steps:
1. Read the conversation to identify the competitor name.
2. Search for the competitor name + "pricing 2025", then + "plans cost", then + "pricing change".
3. Capture all pricing tiers — name, price, what is included.
4. Note any recent price increases, discounts, or free-tier changes.

Return a structured section titled **PRICING INTELLIGENCE** with:
- Pricing Tiers (table: Tier | Price | Key Features)
- Recent Pricing Changes
- Free Trial / Freemium Availability
- Pricing Strategy Assessment (e.g. premium, value, freemium-to-paid)
""",
    tools=[google_search],
)

print("✅ Agent 3 created: pricing_intelligence")

In [ ]:
# ── Agent 4: Report Generator ─────────────────────────────────────────────────
# No tools — it only synthesizes from what the previous agents produced.
report_generator = Agent(
    model=MODEL,
    name="report_generator",
    instruction="""
You are a senior business analyst. Using the MARKET SCAN, SENTIMENT ANALYSIS,
and PRICING INTELLIGENCE sections already gathered in this conversation,
write a final executive intelligence brief.

Do NOT perform any new searches. Synthesize only from what is already in the
conversation above.

Your report must follow this exact structure:

---
## COMPETITIVE INTELLIGENCE BRIEF

### Executive Summary
(3-4 sentences — the most important takeaways)

### Strategic Threats
(What this competitor does well that poses a risk to us)

### Opportunities
(Their weaknesses or gaps we can exploit)

### Recommended Actions
1. ...
2. ...
3. ...

### Competitive Threat Level
Rate: Low / Medium / High — with a one-sentence justification.
---

Tone: professional, concise, C-suite ready.
""",
    tools=[],   # <-- no search tool: synthesizes only
)

print("✅ Agent 4 created: report_generator")

## Section 5 — Wire the Host Agent (SequentialAgent)

`SequentialAgent` runs the sub-agents **in order**.  
Each agent's output lands in the conversation context, so the next agent can see it.

```
market_scanner → sentiment_analyzer → pricing_intelligence → report_generator
```

In [ ]:
host_agent = SequentialAgent(
    name="competitive_intel_host",
    description="Orchestrates market scan, sentiment, pricing, and report generation.",
    sub_agents=[
        market_scanner,
        sentiment_analyzer,
        pricing_intelligence,
        report_generator,
    ],
)

print("✅ Host agent created")
print(f"   Pipeline: {' → '.join(a.name for a in host_agent.sub_agents)}")

## Section 6 — Create the Runner

The `Runner` connects the agent to session memory and handles execution.  
For local testing we use `InMemorySessionService` — no database needed.

In [ ]:
session_service  = InMemorySessionService()
artifact_service = InMemoryArtifactService()
memory_service   = InMemoryMemoryService()

runner = Runner(
    app_name="competitive_intel",
    agent=host_agent,
    artifact_service=artifact_service,
    session_service=session_service,
    memory_service=memory_service,
)

print("✅ Runner ready")

## Section 7 — Helper: Run Agent

A simple async function that sends a query and streams back agent events.  
We print intermediate steps so you can watch each agent working in real time.

In [ ]:
async def run_agent(query: str, session_id: str = "notebook_session") -> str:
    """
    Send a query to the host agent and return the final report.

    ADK 1.9.0 requires an explicit session before run_async — we create it
    here if it does not already exist.
    """
    # Create session if it doesn't exist yet
    existing = await session_service.get_session(
        app_name="competitive_intel",
        user_id="notebook_user",
        session_id=session_id,
    )
    if not existing:
        await session_service.create_session(
            app_name="competitive_intel",
            user_id="notebook_user",
            session_id=session_id,
        )

    message = types.Content(role="user", parts=[types.Part(text=query)])
    final_response = ""
    current_agent  = ""

    async for event in runner.run_async(
        user_id="notebook_user",
        session_id=session_id,
        new_message=message,
    ):
        if hasattr(event, "author") and event.author != current_agent:
            current_agent = event.author
            print(f"\n▶ {current_agent} is working...")

        if event.is_final_response():
            if event.content and event.content.parts:
                final_response = event.content.parts[0].text

    return final_response


print("✅ run_agent() helper defined")

## Section 8 — Test Agent 1 Individually (Market Scanner)

Always test each agent in isolation first before running the full pipeline.  
This makes it easy to debug if something goes wrong.

In [ ]:
single_session_service = InMemorySessionService()

single_runner = Runner(
    app_name="test_market_scanner",
    agent=market_scanner,
    artifact_service=InMemoryArtifactService(),
    session_service=single_session_service,
    memory_service=InMemoryMemoryService(),
)

async def test_single_agent(agent_runner, single_svc, query, session_id="test_session"):
    # ADK 1.9.0: create session explicitly before run_async
    existing = await single_svc.get_session(
        app_name="test_market_scanner",
        user_id="test_user",
        session_id=session_id,
    )
    if not existing:
        await single_svc.create_session(
            app_name="test_market_scanner",
            user_id="test_user",
            session_id=session_id,
        )

    message = types.Content(role="user", parts=[types.Part(text=query)])
    response = ""
    async for event in agent_runner.run_async(
        user_id="test_user",
        session_id=session_id,
        new_message=message,
    ):
        if event.is_final_response() and event.content and event.content.parts:
            response = event.content.parts[0].text
    return response

# Test — scan for Anthropic's latest moves
result = asyncio.run(
    test_single_agent(single_runner, single_session_service, "Analyze competitor: Anthropic")
)
print(result)

## Section 9 — Run the Full Pipeline

Now run all 4 agents in sequence via the host agent.  
**Expected time: 60–120 seconds** (4 agents each making web searches with Gemini 2.5 Pro).

In [ ]:
COMPETITOR = "Anthropic"   # <-- change to any competitor you want to analyze

query = (
    f"Perform a comprehensive competitive intelligence analysis for: {COMPETITOR}. "
    f"Cover recent market moves, brand sentiment, and pricing strategy."
)

print(f"Analyzing: {COMPETITOR}")
print("=" * 60)

report = asyncio.run(run_agent(query, session_id=f"session_{COMPETITOR.lower()}"))

print("\n" + "=" * 60)
print("FINAL REPORT")
print("=" * 60)
print(report)

## Section 10 — Deploy to Vertex AI Agent Engine

Vertex AI Agent Engine is Google's **managed runtime for ADK agents**.  
One command deploys your entire pipeline — no Docker, no server config.

**What you get after deployment:**
- A persistent HTTPS endpoint
- Built-in session management (replaces `InMemorySessionService`)
- Auto-scaling (handles 0 to N concurrent requests)
- IAM-based authentication
- Cloud Logging + Tracing built in

> **Prerequisite:** Enable the Vertex AI API in your GCP project.  
> `gcloud services enable aiplatform.googleapis.com`

In [ ]:
import vertexai
from vertexai.preview import reasoning_engines

# ── Staging bucket ────────────────────────────────────────────────────────────
# Vertex AI Agent Engine uploads your agent code here before deploying.
# The bucket must be in the same region as GOOGLE_CLOUD_LOCATION.
#
# Create it once if it doesn't exist yet:
#   gsutil mb -l us-central1 gs://your-project-id-agent-staging
#
STAGING_BUCKET = f"gs://{GOOGLE_CLOUD_PROJECT}-agent-staging"   # <-- change if needed

vertexai.init(
    project=GOOGLE_CLOUD_PROJECT,
    location=GOOGLE_CLOUD_LOCATION,
    staging_bucket=STAGING_BUCKET,
)

# Wrap the host agent for Agent Engine
adk_app = reasoning_engines.AdkApp(
    agent=host_agent,
    enable_tracing=False,
)

print("Deploying to Vertex AI Agent Engine...")
print("This will take 3-5 minutes...")

remote_agent = reasoning_engines.ReasoningEngine.create(
    adk_app,
    requirements=[
        "google-adk==1.9.0",
        "google-genai",
        "cloudpickle>=3.0.0",
    ],
    display_name="Competitive Intelligence Agent",
    description="Multi-agent competitive intelligence: market scan, sentiment, pricing, report",
)

print(f"\n✅ Deployed successfully!")
print(f"   Resource name : {remote_agent.resource_name}")
print(f"\nSave this resource name — you need it to call the agent.")

## Section 11 — Query the Deployed Agent Engine Endpoint

In [ ]:
# ── Connect to the deployed agent using vertexai.Client ───────────────────────
# This is the correct modern API — it properly binds all streaming methods.
#
# RESOURCE_NAME is set automatically if you ran Section 10 in this session.
# Otherwise paste it directly:
#   RESOURCE_NAME = "projects/YOUR_PROJECT/locations/us-central1/reasoningEngines/1234567890"

RESOURCE_NAME = remote_agent.resource_name   # <-- or paste your resource name here

client = vertexai.Client(
    project=GOOGLE_CLOUD_PROJECT,
    location=GOOGLE_CLOUD_LOCATION,
)

deployed_agent = client.agent_engines.get(name=RESOURCE_NAME)
print(f"✅ Agent loaded: {RESOURCE_NAME}")

In [ ]:
# ── Create a session and stream a query ───────────────────────────────────────
session = deployed_agent.create_session(user_id="prod_user")
print(f"Session created: {session['id']}")
print("=" * 60)

final_report = ""

for event in deployed_agent.stream_query(
    user_id="prod_user",
    session_id=session["id"],
    message="Analyze competitor: OpenAI",
):
    content = event.get("content", {})
    if content:
        for part in content.get("parts", []):
            text = part.get("text", "")
            if text:
                final_report += text
                print(text, end="", flush=True)

print("\n" + "=" * 60)
print("✅ Analysis complete!")